#### Checking and reading text embeddings/similarity
Specifically SMPP dataset

In [9]:
import numpy as np
import pandas as pd
from pathlib import Path

In [19]:
DATA_DIR = Path("..") / "data" / "processed"

embeddings = np.load(DATA_DIR / "SMPP" / "embeddings.npy")
id_map = pd.read_parquet(DATA_DIR / "SMPP" / "embeddings_id_map.parquet")

messages = pd.read_csv(DATA_DIR / "SMPP" / "messages_with_behavioral.csv",
                        usecols=['source','record_id','text'], low_memory=False)
messages['source'] = messages['source'].astype(str)
messages['record_id'] = messages['record_id'].astype(str)
messages['message_key'] = messages['source'] + '|' + messages['record_id']

merged = id_map.merge(messages[['message_key','text']],on='message_key',how='left')
print(embeddings.shape)
print(merged.shape)
merged.head()


(40000, 384)
(40000, 5)


,message_key,source,record_id,timestamp,text
0,SMPP|stg_smpp_20260802_1400#0129183442677630_2...,SMPP,stg_smpp_20260802_1400#0129183442677630_202608...,2026-08-02 14:16:02.699,RM0 MY-HMIRGC: Regency Specialist Hospital - R...
1,SMPP|stg_smpp_20260803_1500#0924328759214656_2...,SMPP,stg_smpp_20260803_1500#0924328759214656_202608...,2026-08-03 15:43:58.206,RM0.00 Bank Of China Debit Card/ACC (9201) Txn...
2,SMPP|stg_smpp_20260802_0800#0353844904234839_2...,SMPP,stg_smpp_20260802_0800#0353844904234839_202608...,2026-08-02 08:07:40.198,RM0 UOB: DuitNow QR was made fr yr A/C ***5016...
3,SMPP|stg_smpp_20260803_1400#0034336015107865_2...,SMPP,stg_smpp_20260803_1400#0034336015107865_202608...,2026-08-03 14:09:10.465,RM0 Apple Store: Items in order W1516814645 we...
4,SMPP|stg_smpp_20260803_1700#0752964723293478_2...,SMPP,stg_smpp_20260803_1700#0752964723293478_202608...,2026-08-03 17:12:04.947,RM0 Parenthood: Never share your OTP! Your ver...


In [30]:
query_idx = 100

query_text = merged.loc[query_idx,'text']
print("Query Message: ", query_text)

sims = embeddings @ embeddings[query_idx]  # matrix vector multiplication to get similarity scores, dot product of every row in embeddings against query vector 
# sims is a 1D array of length N - one number per message, representing the similarity of that message to the query message
top5 = np.argsort(-sims)[1:20] #descending order, most similar messages exclusing itself

print("Most similar other messages:")
for rank,i in enumerate(top5,1):
    print(f"{rank}. (similarity {sims[i]:.3f}) {merged.loc[i, 'text'][:]}")

Query Message:  RM0 MyToyToy OTP is 7877. Do not disclose to anyone.
Most similar other messages:
1. (similarity 0.893) RM0 MyToyToy alert: your OTP is 5641. Do not share it with anyone.
2. (similarity 0.866) RM0 AjimGadget alert: your OTP is 5728. Do not share it with anyone.
3. (similarity 0.865) RM0 MYAIDER: Your OTP is 582521
4. (similarity 0.856) RM0 MyToyToy login OTP: 7753
5. (similarity 0.853) RM0 SOHOJ: Your OTP is 147237. Please do not share your OTP with anyone.
6. (similarity 0.849) RM0 GV RIDE: Your OTP is 8772. Do not share it with anyone.
7. (similarity 0.839) RM0 Your OTP is 651229.
8. (similarity 0.838) RM0 <MYWN77> Your OTP code is : 712967
9. (similarity 0.837) RM0 您的开户申请 OTP 为 351341。切勿向他人透露您的 OTP。
10. (similarity 0.833) RM0 GV RIDE: Your OTP is 1388. Do not share it with anyone.
11. (similarity 0.832) RM0 Your NDE OTP is 14062063. Please do not share this OTP with others.
12. (similarity 0.824) RM0 Kod OTP anda ialah 718172!
13. (similarity 0.824) RM0 MYSMS: Your

In [31]:
for query_idx in [50, 500, 5000, 20000]:
    print("=" * 80)
    print("QUERY:", merged.loc[query_idx, 'text'][:100])
    sims = embeddings @ embeddings[query_idx]
    top3 = np.argsort(-sims)[1:4]
    for i in top3:
        print(f"  -> ({sims[i]:.3f}) {merged.loc[i, 'text'][:100]}")

QUERY: RM0 Refer to the SMS dated 27/07/2026 please be informed that your account has been reassigned to MS
  -> (1.000) RM0 Refer to the SMS dated 27/07/2026 please be informed that your account has been reassigned to MS
  -> (1.000) RM0 Refer to the SMS dated 27/07/2026 please be informed that your account has been reassigned to MS
  -> (1.000) RM0 Refer to the SMS dated 27/07/2026 please be informed that your account has been reassigned to MS
QUERY: RM0 GREATEASTERN <1091018980> will lapse on 08/08/2026. Please pay RM 260.00. Ignore if premium pa
  -> (0.996) RM0 GREATEASTERN <1092683795> will lapse on 08/08/2026. Please pay RM 250.00. Ignore if premium pa
  -> (0.995) RM0 GREATEASTERN <1037086537> will lapse on 08/08/2026. Please pay RM 200.00. Ignore if premium pa
  -> (0.994) RM0 GREATEASTERN <1028729087> will lapse on 09/08/2026. Please pay RM 225.00. Ignore if premium pa
QUERY: RM0 PBB/PIBB: OTP: 204572 Trx amt RM 400.00 @TNG-EWALLET  03Aug26 19:52 for cd ending 9805. D